# 並行混合之策 — DP、TP、PP、SP、CP、EP之合用

> **難度：** 中等 → 上 | **所需時辰：** 約一時辰 | **先修：** 第一至第六篇

大模型之訓練，未嘗獨用一法。
GPT-3 一千七百五十億、LLaMA 七百億、Mixtral 八乘二百二十億，皆**合用多維並行**
——世謂之**三維、四維、五維並行**。

此篇所授：
1. 諸並行維度何者可**正交合用**，何者共享資源
2. 如何將**並行組映射於物理GPU拓撲**（節點、NVLink域）
3. 各配置之**存儲與通信權衡**
4. 擇策之**決策框架**
5. **Megatron-LM之命令行參數**

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import sys, os

# Ensure mp_tutorial is importable
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from mp_tutorial.viz import (
    draw_gpu_topology_grid,
    draw_parallelism_mix_comparison,
    draw_memory_comm_tradeoff,
    draw_decision_flowchart,
    draw_process_group_boxes,
    draw_memory_breakdown_chart,
    GPU_COLORS,
)

## 一、概念與原理

### 六維並行一覽

| 縮寫 | 策略 | 所分者 | 篇目 |
|------|------|--------|------|
| **DP** | 數據並行 | 訓練數據（批次） | 01 |
| **TP** | 張量並行 | 權重矩陣（列/行） | 02 |
| **PP** | 流水線並行 | 模型層（階段） | 03 |
| **SP** | 序列並行 | 沿序列維之激活值 | 04 |
| **CP** | 上下文並行 | 長序列注意力（環形注意力） | 05 |
| **EP** | 專家並行 | MoE專家之副本 | 06 |

六維各分計算之不同軸。其要在於：
**諸維大抵正交**——以其分者各異，故可自由合用。

### 可組合之則

非所有組合皆獨立。其則如下：

| 則 | 釋 |
|------|-----|
| **TP ⊥ PP ⊥ DP** | 張量、流水線、數據並行三者全然正交。總GPU數 = DP × TP × PP |
| **SP共TP之組** | 序列並行與TP共用同一進程組——切分TP已分之激活值。無需額外GPU |
| **CP正交** | 上下文並行另闢一維，用於長序列注意力。可與TP+PP+DP合用 |
| **EP增一維** | 專家並行疊於DP之上——部分DP副本改持不同之專家 |

#### GPU總數公式

$$\text{Total GPUs} = DP \times TP \times PP$$

加SP：無需額外GPU（SP複用TP之組）。

加CP：$\text{Total GPUs} = DP \times TP \times PP \times CP$

加EP：EP細分DP維，故 $DP = DP_{\text{pure}} \times EP$。

### 三維、四維、五維並行

此名示同時啟用之**獨立**並行維度之數：

| 名 | 維度 | 典型用例 |
|------|------|----------|
| **三維並行** | DP × TP × PP | 百億至兩千億參數之模型，多節點集群 |
| **四維並行** | DP × TP × PP × SP（或CP） | 加SP省激活值存儲，或加CP以容極長序列 |
| **五維並行** | DP × TP × PP × SP × EP | 混合專家模型如Mixtral，分專家於諸器 |

> **要旨：** 自簡始（純DP），惟於**存儲不足或需更大規模**時方增維。
> 每增一維，通信之耗亦隨之增。

### 進程組之層次

Megatron-LM等框架中，並行組以層次而組，
猶巢套之迴圈：

```
for dp_rank in range(DP):          # 最外層——跨節點
    for pp_rank in range(PP):       # 中層——跨節點
        for tp_rank in range(TP):   # 最內層——節點內（快速NVLink）
            gpu_id = dp_rank * PP * TP + pp_rank * TP + tp_rank
```

**何以此序？** TP通信量最巨
（每前向/反向步皆需AllReduce），故須置於最速之互連（節點內NVLink）。
PP僅於相鄰階段間傳激活值。
DP每步僅同步一次梯度，故可容較慢之跨節點連接。

In [ ]:
# Demonstrate process group assignment
def assign_gpu_ranks(dp_size, pp_size, tp_size):
    """Show how each GPU gets assigned (TP rank, PP rank, DP rank)."""
    total = dp_size * pp_size * tp_size
    print(f"Total GPUs: {total}  (DP={dp_size} × PP={pp_size} × TP={tp_size})")
    print(f"{'GPU':>5} {'TP rank':>8} {'PP rank':>8} {'DP rank':>8}")
    print("-" * 35)
    for d in range(dp_size):
        for p in range(pp_size):
            for t in range(tp_size):
                gpu = d * pp_size * tp_size + p * tp_size + t
                print(f"{gpu:>5} {t:>8} {p:>8} {d:>8}")

assign_gpu_ranks(dp_size=2, pp_size=2, tp_size=2)

今以巢套之匣觀同一層次——每GPU居於TP組內，
TP組居於PP階段內，PP階段居於DP副本內：

In [ ]:
# Visualize the process group hierarchy as nested boxes
fig, ax = draw_process_group_boxes(dp_size=2, pp_size=2, tp_size=2)
plt.show()

## 二、圖解

### GPU拓撲之映射

混合並行之最要圖解，乃觀**進程組如何映射於物理硬件**。
下示二節點、每節點四GPU之集群，
行三維並行（TP=2, PP=2, DP=2）。

In [ ]:
# 3D parallelism on 2 nodes × 4 GPUs
fig, ax = draw_gpu_topology_grid(
    num_nodes=2, gpus_per_node=4,
    tp_size=2, pp_size=2, dp_size=2
)
plt.show()

**讀圖之法：**
- **色彩** → TP組（共分權重矩陣之GPU，須NVLink）
- **紋理** → PP階段（持模型不同層之GPU）
- **同一TP+PP形態現於不同GPU** → DP副本（同一模型，不同數據）

注意TP組在**同一節點內**（快速互連），
PP與DP則跨節點。

In [ ]:
# Let's try different configurations on the same 8-GPU cluster
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

configs = [
    {"tp": 1, "pp": 1, "dp": 8, "label": "Pure DP (TP=1, PP=1, DP=8)"},
    {"tp": 4, "pp": 1, "dp": 2, "label": "TP-heavy (TP=4, PP=1, DP=2)"},
    {"tp": 2, "pp": 4, "dp": 1, "label": "Deep PP (TP=2, PP=4, DP=1)"},
]

for i, cfg in enumerate(configs):
    fig_i, ax_i = draw_gpu_topology_grid(
        num_nodes=2, gpus_per_node=4,
        tp_size=cfg["tp"], pp_size=cfg["pp"], dp_size=cfg["dp"],
        title=cfg["label"]
    )
    plt.show()

## 三、存儲與通信分析

### 每器之存儲

混合精度訓練（fp16前向，fp32優化器）下每器之存儲：

$$M_{\text{GPU}} = \underbrace{\frac{2P}{TP \cdot PP}}_{\text{fp16 params}} + \underbrace{\frac{12P}{TP \cdot PP}}_{\text{optimizer states}} + \underbrace{A(B, S, H, TP, PP)}_{\text{activations}}$$

其中：
- $P$ = 模型總參數量
- $2P / (TP \cdot PP)$ = 每器之fp16模型權重（每參數2位元組）
- $12P / (TP \cdot PP)$ = Adam優化器狀態（fp32副本 + 動量 + 方差 = 4+4+4位元組）
- $A$ = 激活值存儲（取決於批次大小$B$、序列長度$S$、隱藏維度$H$）

### 每步之通信

| 維度 | 操作 | 每步通信量 | 時機 |
|------|------|-----------|------|
| **TP** | AllReduce | 每層 $2 \times H \times B \times S$（前向+反向） | 每層，前向與反向 |
| **PP** | 點對點 | $B \times S \times H$（激活張量） | 相鄰階段間 |
| **DP** | AllReduce | $2P / TP$（梯度） | 每步一次 |
| **SP** | AllGather + ReduceScatter | $B \times S \times H$ | LayerNorm/Dropout前後 |

In [ ]:
# Compare memory and communication for different mixes on a 70B model
configs = [
    {"tp": 1, "pp": 1, "dp": 8, "label": "Pure DP"},
    {"tp": 2, "pp": 1, "dp": 4, "label": "TP2×DP4"},
    {"tp": 2, "pp": 2, "dp": 2, "label": "TP2×PP2×DP2"},
    {"tp": 4, "pp": 2, "dp": 1, "label": "TP4×PP2"},
]

fig, axes = draw_parallelism_mix_comparison(configs, model_params=70e9)
plt.show()

In [ ]:
# Memory vs Communication trade-off: what happens as we increase TP?
fig, ax = draw_memory_comm_tradeoff(
    vary="tp", vary_range=[1, 2, 4, 8],
    pp_size=1, total_gpus=8, model_params=70e9
)
plt.show()

In [ ]:
# Same trade-off, varying PP instead
fig, ax = draw_memory_comm_tradeoff(
    vary="pp", vary_range=[1, 2, 4, 8],
    tp_size=1, total_gpus=8, model_params=70e9
)
plt.show()

**權衡圖之要旨：**
- 增TP可**減存儲**，然**增通信**（每層皆需AllReduce）
- 增PP可**減存儲**，然**增流水線氣泡**（空閒時間）
- 純DP之**通信最少**，然每器**存儲最多**
- 最佳之配取決於硬件：NVLink速則宜多TP，節點多則宜PP+DP

## 四、大語言模型之應用

### 實際之配置

知名模型所用之並行配置如下：

| 模型 | 參數 | GPU數 | TP | PP | DP | SP | EP | 備註 |
|------|------|-------|----|----|----|----|----|------|
| **GPT-3** | 1750億 | 1024 A100 | 8 | 8 | 16 | — | — | 三維並行，經典Megatron |
| **LLaMA 65B** | 650億 | 2048 A100 | 8 | 4 | 64 | ✓ | — | 啟用SP以省激活值 |
| **LLaMA 70B** | 700億 | 2048 A100 | 8 | 4 | 64 | ✓ | — | GQA減少TP通信開銷 |
| **Mixtral 8×22B** | 1410億 | 512 H100 | 8 | 4 | 4 | ✓ | 4 | 五維並行，MoE模型 |
| **Megatron-Turing NLG** | 5300億 | 2240 A100 | 8 | 35 | 8 | — | — | 極深PP以容巨模型 |

#### 可察之規律

1. **TP = 8**幾為定則——合於DGX節點內八GPU以NVLink互連之架構
2. **PP隨模型深度而增**——愈深之模型需愈多流水線階段
3. **DP填充餘下之GPU**——TP+PP既定，以DP最大化吞吐量
4. **SP恆與TP同行**——免費省激活值存儲，無額外開銷
5. **EP惟用於MoE模型**——於DP維分配專家副本

In [ ]:
# Let's compute the memory breakdown for GPT-3 175B configuration
def compute_memory_breakdown(params_B, tp, pp, dp, hidden=12288, layers=96,
                              seq_len=2048, micro_batch=1, precision_bytes=2):
    """Compute approximate memory per GPU for a given parallelism config.
    
    Args:
        params_B: Total parameters in billions.
        tp, pp, dp: Parallelism sizes.
        hidden: Hidden dimension.
        layers: Total transformer layers.
        seq_len: Sequence length.
        micro_batch: Micro-batch size per GPU.
        precision_bytes: Bytes per parameter (2 for fp16).
    
    Returns:
        Dict with memory breakdown in GB.
    """
    P = params_B * 1e9
    params_per_gpu = P / (tp * pp)
    
    # Model weights (fp16)
    weight_mem = precision_bytes * params_per_gpu
    
    # Optimizer states (Adam: fp32 copy + momentum + variance = 12 bytes/param)
    optimizer_mem = 12 * params_per_gpu
    
    # Gradients (fp16)
    gradient_mem = precision_bytes * params_per_gpu
    
    # Activations (rough estimate: 2 * seq_len * hidden * layers/pp * micro_batch * 2 bytes)
    layers_per_stage = layers // pp
    activation_mem = 2 * seq_len * (hidden // tp) * layers_per_stage * micro_batch * precision_bytes
    
    to_gb = 1 / (1024**3)
    breakdown = {
        "Weights (fp16)": weight_mem * to_gb,
        "Optimizer (fp32)": optimizer_mem * to_gb,
        "Gradients (fp16)": gradient_mem * to_gb,
        "Activations": activation_mem * to_gb,
    }
    breakdown["Total"] = sum(breakdown.values())
    return breakdown

# GPT-3 175B: TP=8, PP=8, DP=16 on 1024 GPUs
mem = compute_memory_breakdown(175, tp=8, pp=8, dp=16, hidden=12288, layers=96)
print("GPT-3 175B — Memory per GPU (TP=8, PP=8, DP=16):")
print(f"{'Component':<25} {'GB':>8}")
print("-" * 35)
for k, v in mem.items():
    print(f"{k:<25} {v:>8.1f}")
print(f"\n→ Fits in 80GB A100? {'Yes ✓' if mem['Total'] < 80 else 'No ✗'}")

In [ ]:
# Compare: what if we used different configs for the same 175B model?
configs_175b = [
    ("TP=8, PP=8", 8, 8),
    ("TP=8, PP=4", 8, 4),
    ("TP=4, PP=4", 4, 4),
    ("TP=8, PP=1", 8, 1),
]

print(f"{'Config':<20} {'Mem/GPU (GB)':>15} {'Fits 80GB?':>12}")
print("-" * 50)
for label, tp, pp in configs_175b:
    dp = max(1, 1024 // (tp * pp))
    mem = compute_memory_breakdown(175, tp=tp, pp=pp, dp=dp, hidden=12288, layers=96)
    fits = "Yes ✓" if mem['Total'] < 80 else "No ✗"
    print(f"{label:<20} {mem['Total']:>15.1f} {fits:>12}")

今以堆疊柱狀圖示同一比較——何者佔存儲之大宗，
何種配置可入GPU之存儲，一目了然：

In [ ]:
# Stacked memory breakdown: GPT-3 175B with different parallelism configs
configs_viz = [
    {"tp": 8, "pp": 8, "dp": 16, "label": "TP8×PP8×DP16\n(actual GPT-3)"},
    {"tp": 8, "pp": 4, "dp": 32, "label": "TP8×PP4×DP32"},
    {"tp": 4, "pp": 4, "dp": 64, "label": "TP4×PP4×DP64"},
    {"tp": 8, "pp": 1, "dp": 128, "label": "TP8×PP1×DP128"},
]

fig, ax = draw_memory_breakdown_chart(
    configs_viz, model_params_B=175, hidden=12288, layers=96,
    gpu_memory_gb=80,
    title="GPT-3 175B — Memory Breakdown per GPU"
)
plt.show()

## 五、實操：決策框架

### 並行混合之決策流程圖

依此逐步之流程，以定並行之配置。
下圖引君自「始」至推薦之混合策略：

In [ ]:
# Visual decision flowchart
fig, ax = draw_decision_flowchart()
plt.show()

In [ ]:
def recommend_parallelism(model_params_B, num_gpus, gpu_memory_gb=80,
                           gpus_per_node=8, seq_len=2048, is_moe=False, num_experts=0):
    """Recommend a parallelism configuration based on model and cluster specs.
    
    Args:
        model_params_B: Total parameters in billions.
        num_gpus: Total available GPUs.
        gpu_memory_gb: Memory per GPU in GB.
        gpus_per_node: GPUs per node.
        seq_len: Training sequence length.
        is_moe: Whether this is a MoE model.
        num_experts: Number of experts (if MoE).
    
    Returns:
        Dict with recommended config and reasoning.
    """
    P = model_params_B * 1e9
    # Memory needed for model + optimizer (14 bytes/param for fp16 + Adam)
    model_memory_gb = (14 * P) / (1024**3)
    
    reasoning = []
    tp, pp, dp, sp, cp, ep = 1, 1, num_gpus, False, 1, 1
    
    # Step 1: Does it fit on 1 GPU?
    if model_memory_gb <= gpu_memory_gb * 0.8:  # 80% headroom for activations
        reasoning.append(f"Model ({model_memory_gb:.1f}GB) fits on 1 GPU → Pure DP")
        return {"tp": 1, "pp": 1, "dp": num_gpus, "sp": False, "cp": 1, "ep": 1,
                "reasoning": reasoning}
    
    # Step 2: Add TP (within node)
    tp = min(gpus_per_node, num_gpus)
    mem_with_tp = model_memory_gb / tp
    reasoning.append(f"Model too large ({model_memory_gb:.1f}GB) → Add TP={tp}")
    reasoning.append(f"  Memory per GPU with TP: {mem_with_tp:.1f}GB")
    
    if mem_with_tp <= gpu_memory_gb * 0.8:
        dp = num_gpus // tp
        reasoning.append(f"Fits with TP={tp} → TP + DP (DP={dp})")
        return {"tp": tp, "pp": 1, "dp": dp, "sp": True, "cp": 1, "ep": 1,
                "reasoning": reasoning}
    
    # Step 3: Add PP
    pp = 1
    while model_memory_gb / (tp * pp) > gpu_memory_gb * 0.7:
        pp *= 2
        if tp * pp > num_gpus:
            break
    dp = num_gpus // (tp * pp)
    sp = True  # Always enable SP when TP > 1
    mem_final = model_memory_gb / (tp * pp)
    reasoning.append(f"Still too large → Add PP={pp}")
    reasoning.append(f"  Memory per GPU: {mem_final:.1f}GB")
    reasoning.append(f"  Final: TP={tp}, PP={pp}, DP={dp}, SP={sp}")
    
    # Step 4: Long sequences → CP
    if seq_len > 8192 and dp > 1:
        cp = min(4, dp)
        dp = dp // cp
        reasoning.append(f"Long seq ({seq_len}) → Add CP={cp}, DP reduced to {dp}")
    
    # Step 5: MoE → EP
    if is_moe and num_experts > 1 and dp > 1:
        ep = min(num_experts, dp)
        dp = dp // ep
        reasoning.append(f"MoE ({num_experts} experts) → Add EP={ep}, DP reduced to {dp}")
    
    return {"tp": tp, "pp": pp, "dp": dp, "sp": sp, "cp": cp, "ep": ep,
            "reasoning": reasoning}

In [ ]:
# Example 1: LLaMA 7B on 8 GPUs
result = recommend_parallelism(7, num_gpus=8, gpu_memory_gb=80)
print("=" * 50)
print("LLaMA 7B on 8× A100-80GB")
print("=" * 50)
for line in result["reasoning"]:
    print(f"  {line}")
print(f"\n  → Config: TP={result['tp']}, PP={result['pp']}, DP={result['dp']}")

In [ ]:
# Example 2: LLaMA 70B on 64 GPUs
result = recommend_parallelism(70, num_gpus=64, gpu_memory_gb=80)
print("=" * 50)
print("LLaMA 70B on 64× A100-80GB")
print("=" * 50)
for line in result["reasoning"]:
    print(f"  {line}")
print(f"\n  → Config: TP={result['tp']}, PP={result['pp']}, DP={result['dp']}, SP={result['sp']}")

In [ ]:
# Example 3: GPT-3 175B on 1024 GPUs
result = recommend_parallelism(175, num_gpus=1024, gpu_memory_gb=80)
print("=" * 50)
print("GPT-3 175B on 1024× A100-80GB")
print("=" * 50)
for line in result["reasoning"]:
    print(f"  {line}")
print(f"\n  → Config: TP={result['tp']}, PP={result['pp']}, DP={result['dp']}, SP={result['sp']}")

In [ ]:
# Example 4: Mixtral 8×22B (MoE) on 512 GPUs with long sequences
result = recommend_parallelism(141, num_gpus=512, gpu_memory_gb=80,
                                seq_len=32768, is_moe=True, num_experts=8)
print("=" * 50)
print("Mixtral 8×22B on 512× H100-80GB (seq=32k)")
print("=" * 50)
for line in result["reasoning"]:
    print(f"  {line}")
print(f"\n  → Config: TP={result['tp']}, PP={result['pp']}, DP={result['dp']}, "
      f"SP={result['sp']}, CP={result['cp']}, EP={result['ep']}")

### 交互式存儲計算器

修改下列參數，觀存儲與通信之變化。

In [ ]:
# === MODIFY THESE PARAMETERS ===
MODEL_PARAMS_B = 70      # Model size in billions
TOTAL_GPUS     = 64      # Total GPUs in your cluster
GPU_MEMORY_GB  = 80      # Memory per GPU (e.g., 40, 80)
GPUS_PER_NODE  = 8       # GPUs per node

# Try different configs
my_configs = [
    {"tp": 1, "pp": 1, "dp": 64},   # Pure DP
    {"tp": 8, "pp": 1, "dp": 8},    # TP only
    {"tp": 8, "pp": 2, "dp": 4},    # TP + PP
    {"tp": 8, "pp": 4, "dp": 2},    # More PP
]

print(f"Model: {MODEL_PARAMS_B}B params | Cluster: {TOTAL_GPUS} GPUs | GPU: {GPU_MEMORY_GB}GB")
print("=" * 70)
print(f"{'Config':<20} {'Mem/GPU':>10} {'Fits?':>8} {'DP throughput':>15}")
print("-" * 70)

for cfg in my_configs:
    tp, pp, dp = cfg['tp'], cfg['pp'], cfg['dp']
    if tp * pp * dp != TOTAL_GPUS:
        print(f"TP{tp}×PP{pp}×DP{dp}  — INVALID (product ≠ {TOTAL_GPUS})")
        continue
    mem = compute_memory_breakdown(MODEL_PARAMS_B, tp=tp, pp=pp, dp=dp)
    fits = "Yes ✓" if mem['Total'] < GPU_MEMORY_GB else "No ✗"
    # DP throughput: higher DP = more data parallelism = higher throughput
    # But PP adds bubble overhead: efficiency ≈ M/(M+P-1) where M=microbatches, P=stages
    bubble_eff = 1.0 if pp == 1 else 8 / (8 + pp - 1)  # assume 8 microbatches
    throughput = f"DP={dp}, eff={bubble_eff:.0%}"
    label = f"TP{tp}×PP{pp}×DP{dp}"
    print(f"{label:<20} {mem['Total']:>8.1f}GB {fits:>8} {throughput:>15}")

## 六、Megatron-LM參考

Megatron-LM以命令行參數配置並行之混合。關鍵參數如下：

| 參數 | 默認 | 釋義 |
|------|------|------|
| `--tensor-model-parallel-size` | 1 | TP度——於節點內分切權重矩陣 |
| `--pipeline-model-parallel-size` | 1 | PP度——分模型層為流水線階段 |
| `--data-parallel-size` | 自動 | DP度——自動算為 total_gpus / (TP × PP) |
| `--sequence-parallel` | 關 | 啟用SP——沿序列維分激活值（須TP > 1） |
| `--context-parallel-size` | 1 | CP度——以環形注意力分長序列 |
| `--num-experts` | — | 每層之MoE專家數 |
| `--expert-model-parallel-size` | 1 | EP度——分專家於諸器 |
| `--num-layers-per-virtual-pipeline-stage` | — | 交錯PP——每器持多段非連續層，以減氣泡 |
| `--micro-batch-size` | — | 每器微批次——愈小則激活值存儲愈少，然氣泡愈多 |
| `--global-batch-size` | — | 全局批次 = micro × DP × gradient_acc_steps |

In [ ]:
# Example Megatron-LM launch commands for different scales

megatron_examples = {
    "GPT-3 175B (1024 GPUs)": """
torchrun --nproc_per_node=8 --nnodes=128 \\
    pretrain_gpt.py \\
    --tensor-model-parallel-size 8 \\
    --pipeline-model-parallel-size 8 \\
    --micro-batch-size 1 \\
    --global-batch-size 1536 \\
    --sequence-parallel \\
    --num-layers 96 \\
    --hidden-size 12288 \\
    --num-attention-heads 96
""",
    "LLaMA 70B (64 GPUs)": """
torchrun --nproc_per_node=8 --nnodes=8 \\
    pretrain_gpt.py \\
    --tensor-model-parallel-size 8 \\
    --pipeline-model-parallel-size 2 \\
    --micro-batch-size 1 \\
    --global-batch-size 256 \\
    --sequence-parallel \\
    --num-layers 80 \\
    --hidden-size 8192 \\
    --num-attention-heads 64
""",
    "Mixtral 8×22B MoE (512 GPUs)": """
torchrun --nproc_per_node=8 --nnodes=64 \\
    pretrain_gpt.py \\
    --tensor-model-parallel-size 8 \\
    --pipeline-model-parallel-size 4 \\
    --expert-model-parallel-size 4 \\
    --num-experts 8 \\
    --micro-batch-size 1 \\
    --global-batch-size 128 \\
    --sequence-parallel \\
    --context-parallel-size 2 \\
    --seq-length 32768
""",
}

for name, cmd in megatron_examples.items():
    print(f"# {name}")
    print(cmd)

## 七、總結與延讀

### 要旨

1. **並行維度大抵正交**：DP × TP × PP自由組合。SP附於TP。EP細分DP。

2. **TP置最內層**（節點內，以NVLink連），蓋其每層皆通信。PP與DP可用較慢之跨節點連接。

3. **自簡始，循需增維**：純DP → 模型不入則加TP → 甚大模型加PP → SP免費啟用 → 長序列加CP → MoE加EP。

4. **存儲隨之遞減**：$M \propto \frac{P}{TP \times PP}$，然通信隨每增之維而增。

5. **流水線氣泡**乃PP之主耗：效率 ≈ $\frac{M}{M + P - 1}$，M為微批次數，P為流水線階段數。

### 要式

| 式 | 義 |
|------|------|
| $\text{Total GPUs} = DP \times TP \times PP$ | GPU總數公式 |
| $M_{\text{params/GPU}} = \frac{2P}{TP \times PP}$ | 每器fp16模型存儲 |
| $M_{\text{optim/GPU}} = \frac{12P}{TP \times PP}$ | 每器Adam優化器存儲 |
| $\eta_{\text{pipeline}} = \frac{M}{M + P - 1}$ | 流水線效率（M=微批次數，P=階段數） |

### 延讀

- [Megatron-LM Paper: Efficient Large-Scale Language Model Training on GPU Clusters](https://arxiv.org/abs/2104.04473)
- [Reducing Activation Recomputation in Large Transformer Models (SP)](https://arxiv.org/abs/2205.05198)
- [Ring Attention with Blockwise Transformers (CP)](https://arxiv.org/abs/2310.01889)
- [Switch Transformers (MoE/EP)](https://arxiv.org/abs/2101.03961)
- [DeepSpeed ZeRO](https://arxiv.org/abs/1910.02054)

### 後續

六維並行及其合用之法至此已畢！
進階之題如激活值檢查點、梯度累積、混合精度訓練之細節，
請參 `notebooks/07-advanced-topics/`。